# Lab 03 — LLM-as-a-Judge Eval Harness from Scratch

**Pairs with:** [Course 13 · LLM Evaluation & Quality](https://psssnikhil.github.io/ai-engineering-handbook/production/module-19-llm-evaluation-quality/)

In this lab, you will build a production-grade **LLM-as-a-Judge Evaluation Harness** from scratch in pure Python with **no frameworks**.

You will implement:
1. **G-Eval Rubric Scoring** — Structured JSON evaluation criteria for Faithfulness, Relevance, and Completeness.
2. **Batch Evaluation Runner** — Running evaluation cases against a golden dataset.
3. **Aggregate Metric Reporting** — Calculating pass rates, mean score distribution, and regression metrics.
4. **CI/CD Quality Gates** — Asserting performance thresholds for automated deployment pipelines.

```
Golden Dataset (Query, Context, Output) ──► Judge Model (Claude) ──► JSON Score Card ──► CI/CD Gate
```

**Prerequisites:** `pip install -r requirements.txt` and `ANTHROPIC_API_KEY` set in your environment.

In [ ]:
import os
import json
import anthropic
from typing import List, Dict, Any

client = anthropic.Anthropic()
JUDGE_MODEL = "claude-3-5-sonnet-20241022"

## Step 1: Golden Dataset Setup

A golden evaluation dataset contains reference queries, retrieved context, generated outputs, and ground-truth answers.

In [ ]:
GOLDEN_DATASET = [
    {
        "id": "case_01",
        "query": "What is the refund window for hardware items?",
        "context": "Hardware items may be returned within 30 days of delivery with original receipt. Software downloads are non-refundable.",
        "actual_output": "Hardware items can be returned within 30 days of delivery as long as you have the original receipt."
    },
    {
        "id": "case_02",
        "query": "Can I get a refund on a software download?",
        "context": "Hardware items may be returned within 30 days of delivery with original receipt. Software downloads are non-refundable.",
        "actual_output": "Yes, software downloads can be refunded within 14 days."
    }
]

## Step 2: Define LLM-as-a-Judge Rubric & System Prompt

We prompt the judge LLM to evaluate **Faithfulness** (no hallucination relative to context) and **Relevance** on a 1-5 scale, returning strict JSON.

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert AI Evaluation Judge. Evaluate the provided model output against the query and retrieved context.

Criteria:
1. Faithfulness (1-5): Is the answer entirely supported by the context without hallucinations or false claims?
2. Relevance (1-5): Does the answer directly address the user's query?

You must return ONLY a JSON object in this exact format:
{
  "faithfulness_score": <1-5>,
  "relevance_score": <1-5>,
  "reasoning": "<1-2 sentences explaining scores>"
}
"""

## Step 3: Implement Single Case Evaluation

In [ ]:
def evaluate_case(test_case: Dict[str, Any]) -> Dict[str, Any]:
    user_content = f"""Query: {test_case['query']}
Retrieved Context: {test_case['context']}
Actual Output: {test_case['actual_output']}
"""
    
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=300,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_content}]
    )
    
    judge_raw = response.content[0].text.strip()
    try:
        eval_result = json.loads(judge_raw)
    except Exception as e:
        eval_result = {"faithfulness_score": 1, "relevance_score": 1, "reasoning": f"Parse Error: {e}"}
        
    eval_result["id"] = test_case["id"]
    return eval_result

## Step 4: Batch Evaluation & CI Quality Gate Report

We run evaluations over the dataset, compute mean scores, and assert pass criteria (e.g. Faithfulness >= 4.0).

In [ ]:
def run_eval_suite(dataset: List[Dict[str, Any]], min_faithfulness: float = 4.0):
    results = []
    print(f"=== Running Eval Suite over {len(dataset)} cases ===\n")
    
    for case in dataset:
        res = evaluate_case(case)
        results.append(res)
        print(f"Case ID: {res['id']} | Faithfulness: {res['faithfulness_score']}/5 | Relevance: {res['relevance_score']}/5")
        print(f"Reasoning: {res['reasoning']}\n")
        
    mean_faithfulness = sum(r["faithfulness_score"] for r in results) / len(results)
    mean_relevance = sum(r["relevance_score"] for r in results) / len(results)
    
    print(f"=== SUMMARY REPORT ===")
    print(f"Mean Faithfulness: {mean_faithfulness:.2f} / 5.0")
    print(f"Mean Relevance:    {mean_relevance:.2f} / 5.0")
    
    # CI/CD Quality Gate Assertion
    if mean_faithfulness >= min_faithfulness:
        print("\n✅ PASSED CI Quality Gate!")
        return True
    else:
        print(f"\n❌ FAILED CI Quality Gate: Faithfulness {mean_faithfulness:.2f} < threshold {min_faithfulness:.2f}")
        return False

## Step 5: Test the Harness

Notice how Case 02 contains a hallucination (claiming software downloads are refundable). The judge detects this and fails the quality gate.

In [ ]:
run_eval_suite(GOLDEN_DATASET, min_faithfulness=4.0)